# ADVC — Local Jupyter Notebook (UoM H100 Workstation)

**Environment:** 10 GB VRAM H100 slice · 1 core Intel Xeon Platinum 8480+ · 28 GB RAM

Run cells top-to-bottom in order. Each phase is resumable — re-running a cell skips already-completed rows.

**Config changes already applied in `configs/base.yaml`:**
- `eval.batch_size`: 64 → 32 (8 GB VRAM)
- `defense.batch_size`: 32 → 16 (8 GB VRAM)
- `eval.num_workers`: 2 → 0 (1 CPU core — avoids DataLoader deadlock)

In [ ]:
# Cell 0 — Set working directory to ADVC repo root (run this first, always)
import os
from pathlib import Path

_cwd = Path().resolve()

# Check common locations in order — Trash paths are explicitly excluded
_candidates = [
    Path('/ADVC'),                 # UoM JupyterHub location
    Path('/root/ADVC'),
    Path('/root/data/ADVC'),
    Path('/root/data'),
    Path.home() / 'ADVC',
    _cwd,
    _cwd.parent,
    _cwd.parent.parent,
]

# Filter out any path that lives inside a Trash directory
def _is_trash(p: Path) -> bool:
    return 'Trash' in str(p)

for candidate in _candidates:
    if _is_trash(candidate):
        continue
    if (candidate / 'configs' / 'base.yaml').exists():
        os.chdir(str(candidate))
        print('Working directory set to: ' + str(candidate))
        break
else:
    print('WARNING: Could not find repo root automatically.')
    print('Current cwd: ' + str(_cwd))
    print('Manually run:  os.chdir("/ADVC")  then re-run this cell.')

print('cwd is now: ' + os.getcwd())

In [ ]:
# Cell 0b — Pull latest code from GitHub (run after Cell 0, any time you want updates)
import subprocess

result = subprocess.run(
    ['git', 'pull', '--ff-only'],
    capture_output=True, text=True,
)
print(result.stdout.strip() or '(no output)')
if result.returncode != 0:
    print('git pull failed:')
    print(result.stderr.strip())
else:
    print('Repository is up to date.')

## Cell 1 — Verify GPU

Confirm the H100 GPU is available before starting long experiments.

In [ ]:
# Cell 1 — Verify GPU
import torch

print('CUDA available : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    print('GPU name       : ' + gpu_name)
    print(f'Free VRAM      : {free_vram/1e9:.1f} GB  /  {total_vram/1e9:.1f} GB total')
    if 'H100' not in gpu_name:
        print()
        print('WARNING: Expected H100 but got: ' + gpu_name)
        print('Verify your runtime is assigned to the H100 slice.')
    else:
        print()
        print('H100 confirmed. Ready to proceed.')
else:
    print()
    print('WARNING: No CUDA GPU detected. Check your JupyterHub resource allocation.')

## Cell 2 — Install Dependencies

Installs required packages. Also checks whether the pre-installed PyTorch can see the GPU — if not (CUDA driver mismatch), it reinstalls `torch+torchvision` for `cu121`, which is compatible with the CUDA 12.3 driver on this server.

**If torch is reinstalled, you must restart the kernel before continuing.**

In [ ]:
# Cell 2 — Install Dependencies
import subprocess, sys, torch

# ── Check PyTorch version and GPU visibility ───────────────────────────────────
_cuda_ok = torch.cuda.is_available()
_version = torch.__version__
print(f'Current PyTorch : {_version}')
print(f'CUDA available  : {_cuda_ok}')

# transformers requires torch >= 2.4.0 for bitsandbytes INT8/INT4 to work.
# cu121 wheels exist for 2.4.x and are compatible with CUDA driver 12.3.
_major, _minor = int(_version.split('.')[0]), int(_version.split('.')[1])
_needs_upgrade = (not _cuda_ok) or (_major < 2) or (_major == 2 and _minor < 4)

if _needs_upgrade:
    print()
    if not _cuda_ok:
        print('PyTorch cannot see the GPU — reinstalling.')
    else:
        print(f'PyTorch {_version} < 2.4 — transformers will disable torch, breaking INT8/INT4.')
        print('Upgrading to torch 2.4.1+cu121 ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch==2.4.1+cu121', 'torchvision==0.19.1+cu121',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print('torch install failed:\n' + result.stderr[-2000:])
    else:
        print('torch 2.4.1+cu121 installed.')
        print('IMPORTANT: Restart the kernel now, then re-run from Cell 0.')
else:
    print(f'PyTorch {_version} >= 2.4 and CUDA OK — skipping torch reinstall.')

# ── Install remaining packages ────────────────────────────────────────────────
packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml',
            'tqdm', 'accelerate', 'huggingface_hub',
            'transformers>=4.44.0']   # 4.44+ supports torch 2.4

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('pip error:\n' + result.stderr[-2000:])
else:
    print('All packages ready: ' + ', '.join(packages))

## Cell 3 — Extract Imagenette archive and Set Paths

**Edit `ARCHIVE_PATH` to point to your uploaded imagenette file.**

Supports both `.zip` and `.tgz` / `.tar.gz` archives.

The archive will be extracted into `data/` inside the project root.
Label remapping (imagenette 10-class → ImageNet-1k indices) is already handled by the experiment scripts.

In [ ]:
# Cell 3 — Extract Imagenette archive and Set Paths
import os, sys, yaml, zipfile, tarfile
from pathlib import Path

# ── EDIT THIS LINE — full path to the archive you uploaded ────────────────────
ARCHIVE_PATH = '/path/to/imagenette2-320.tgz'   # .tgz, .tar.gz, or .zip all work
# ─────────────────────────────────────────────────────────────────────────────

# Resolve PROJECT_ROOT (this notebook lives at ADVC/notebooks/)
_THIS_NOTEBOOK = Path(os.path.abspath('notebooks/run_phases_local.ipynb'))
PROJECT_ROOT   = str(_THIS_NOTEBOOK.parent.parent)

if not os.path.exists(os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')):
    _search = Path(os.getcwd())
    for candidate in [_search, _search.parent, _search.parent.parent]:
        if (candidate / 'configs' / 'base.yaml').exists():
            PROJECT_ROOT = str(candidate)
            break

print('Project root : ' + PROJECT_ROOT)
assert os.path.exists(os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')), \
    'Cannot find configs/base.yaml — make sure you cloned the repo and opened the notebook from inside it.'

EXTRACT_DIR = os.path.join(PROJECT_ROOT, 'data')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working dir  : ' + os.getcwd())

# ── Extract ───────────────────────────────────────────────────────────────────
if not os.path.exists(ARCHIVE_PATH):
    raise FileNotFoundError(
        f'Archive not found: {ARCHIVE_PATH}\n'
        'Upload your imagenette archive via the Jupyter file browser, then update ARCHIVE_PATH above.'
    )

print(f'Extracting {ARCHIVE_PATH} ...')
os.makedirs(EXTRACT_DIR, exist_ok=True)

_name = ARCHIVE_PATH.lower()
if _name.endswith('.zip'):
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
elif _name.endswith(('.tgz', '.tar.gz', '.tar.bz2', '.tar.xz')):
    with tarfile.open(ARCHIVE_PATH, 'r:*') as tf:
        tf.extractall(EXTRACT_DIR)
else:
    raise ValueError(f'Unrecognised archive format: {ARCHIVE_PATH}\nExpected .zip, .tgz, or .tar.gz')

print('Extraction complete.')

# ── Find the extracted root (handles imagenette2-320/, imagenette2/, etc.) ────
candidates = [
    d for d in os.listdir(EXTRACT_DIR)
    if d.startswith('imagenette') and os.path.isdir(os.path.join(EXTRACT_DIR, d))
]
if not candidates:
    raise RuntimeError('Could not find an imagenette* folder inside ' + EXTRACT_DIR)
IMAGENETTE_ROOT = os.path.join(EXTRACT_DIR, candidates[0])
print('Imagenette root: ' + IMAGENETTE_ROOT)

# ── Write paths into configs/base.yaml ────────────────────────────────────────
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs', 'base.yaml')
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['val_dir']   = os.path.join(IMAGENETTE_ROOT, 'val')
cfg['dataset']['train_dir'] = os.path.join(IMAGENETTE_ROOT, 'train')

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print()
print('configs/base.yaml updated:')
print('  val_dir  : ' + cfg['dataset']['val_dir'])
print('  train_dir: ' + cfg['dataset']['train_dir'])

# ── Verify ────────────────────────────────────────────────────────────────────
ok = True
for label, path in [('val', cfg['dataset']['val_dir']), ('train', cfg['dataset']['train_dir'])]:
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f'  OK   {label}/ found  ({n} class folders)')
    else:
        print(f'  MISSING: {path}')
        ok = False

if ok:
    print()
    print('Imagenette ready. Proceed to Cell 4.')
else:
    print()
    print('ERROR: extraction may have failed — check the zip contents and re-run.')

## Cell 4 — Create Output Directories

In [ ]:
# Cell 4 — Create Output Directories
import os

dirs = [
    'results',
    'results/checkpoints/at',
    'results/checkpoints/atkd',
    'results/figures',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print('Ready: ' + d)

print()
print('All directories ready.')

## Cell 5 — Smoke Test: Load Model

Load FP32 DeiT-S and run one forward pass to confirm the environment is working before starting long experiments.

> **bitsandbytes on Linux:** INT8/INT4 require `bitsandbytes>=0.41` with CUDA support. If `load_model` fails for int8/int4, confirm with `python -m bitsandbytes` that CUDA kernels loaded.

In [ ]:
# Cell 5 — Smoke Test: Load FP32 DeiT-S
import torch
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading FP32 DeiT-S ...')
model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    # Handle HuggingFace dataclass output
    if hasattr(out, 'logits'):
        out = out.logits

print(f'Output shape: {out.shape}  (expected [2, 1000])')
print(f'VRAM used   : {torch.cuda.memory_allocated()/1e6:.0f} MB')

del model, dummy
torch.cuda.empty_cache()
print()
print('Smoke test passed. Environment is ready.')

## Cell 6 — Run Phase 1: No Defense (Baseline)

Sweeps INT8/INT4 × FGSM/PGD/Patch. Writes 6 rows to `results/phase1_results.csv`.

**Estimated time on H100 8GB slice:** ~45–60 min (patch attack dominates).

The cell is fully resumable — kill and re-run at any point.

In [ ]:
# Cell 5b — Clear VRAM and set HF_TOKEN before running experiments
import torch, os, gc

# Free any leftover allocations from smoke test or previous runs
gc.collect()
torch.cuda.empty_cache()
free, total = [x / 1e9 for x in torch.cuda.mem_get_info(0)]
print(f'VRAM free: {free:.1f} GB / {total:.1f} GB')
if free < 3.0:
    print('WARNING: Less than 3 GB free — restart the kernel before running experiments.')
else:
    print('VRAM OK.')

# Set HF_TOKEN to avoid rate-limit warnings and enable authenticated downloads.
# Get your token from https://huggingface.co/settings/tokens
# Paste it below between the quotes.
HF_TOKEN = ''   # ← paste your HuggingFace token here

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN set.')
else:
    print('WARNING: HF_TOKEN not set — INT8/INT4 model downloads may be rate-limited or slow.')

In [ ]:
# Cell 6 — Run Phase 1: No Defense
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase1.py')
RESULTS_FILE = os.path.join('results', 'phase1_results.csv')

print('=' * 60)
print('Phase 1: No-defense baseline sweep')
print('=' * 60)

env = {**os.environ}  # inherit full env including HF_TOKEN

proc = subprocess.Popen(
    [sys.executable, SCRIPT, '--model', 'deit_small'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Phase 1] Exited with code ' + str(proc.returncode))
if os.path.exists(RESULTS_FILE):
    print('[Phase 1] Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 1] WARNING: results file not found — check output above for errors.')

## Cell 7 — Run Phase 2a: Adversarial Training (AT) Defense

Fine-tunes compressed DeiT-S with AT (7 epochs each), then evaluates all 3 attacks.
Writes 6 rows to `results/phase2_at_results.csv`.

**Estimated time on H100 8GB slice:** ~1.5–2 hr (7 epochs × 2 compression levels × training + eval).

Checkpoints saved after every epoch to `results/checkpoints/at/`.

In [ ]:
# Cell 7 — Run Phase 2a: AT Defense (all compression levels)
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase2_at.py')
RESULTS_FILE = os.path.join('results', 'phase2_at_results.csv')

env = {**os.environ}  # inherit full env including HF_TOKEN

for compression in ['int8', 'int4']:
    print('=' * 60)
    print(f'Phase 2a: AT defense — {compression}')
    print('=' * 60)

    proc = subprocess.Popen(
        [sys.executable, SCRIPT, '--compression', compression],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print(f'[Phase 2a {compression}] Exited with code ' + str(proc.returncode))
    print()

if os.path.exists(RESULTS_FILE):
    print('[Phase 2a] All done. Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 2a] WARNING: results file not found — check output above for errors.')

## Cell 7b — (Optional) Run a Single Compression Level for Phase 2a

Use this if you want to run one level at a time (e.g. to check results before continuing).
Set `COMPRESSION` to `int8` or `int4`.

In [ ]:
# Cell 7b — Phase 2a: Single Compression Level
import subprocess, sys, os

COMPRESSION = 'int8'   # change to 'int4'
SCRIPT = os.path.join('experiments', 'eval_phase2_at.py')

print(f'Running Phase 2a for compression={COMPRESSION} only ...')

proc = subprocess.Popen(
    [sys.executable, SCRIPT, '--compression', COMPRESSION],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print('[Phase 2a single] Exited with code ' + str(proc.returncode))

## Cell 8 — Run Phase 2b: AT+KD Defense

Same structure as Phase 2a but adds a frozen FP32 teacher for KL-divergence supervision.
Both teacher and student are loaded simultaneously — peak VRAM ~2.5 GB (fits in 8 GB slice).

Writes 6 rows to `results/phase2_atkd_results.csv`. Checkpoints in `results/checkpoints/atkd/`.

In [ ]:
# Cell 8 — Run Phase 2b: AT+KD Defense (all compression levels)
import subprocess, sys, os

SCRIPT = os.path.join('experiments', 'eval_phase2_atkd.py')
RESULTS_FILE = os.path.join('results', 'phase2_atkd_results.csv')

env = {**os.environ}  # inherit full env including HF_TOKEN

for compression in ['int8', 'int4']:
    print('=' * 60)
    print(f'Phase 2b: AT+KD defense — {compression}')
    print('=' * 60)

    proc = subprocess.Popen(
        [sys.executable, SCRIPT, '--compression', compression],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print(f'[Phase 2b {compression}] Exited with code ' + str(proc.returncode))
    print()

if os.path.exists(RESULTS_FILE):
    print('[Phase 2b] All done. Results saved to ' + RESULTS_FILE)
else:
    print('[Phase 2b] WARNING: results file not found — check output above for errors.')

## Cell 10 — Preview All Results

Load all completed CSVs and display as formatted tables. Run at any point to see progress.

In [ ]:
# Cell 10 — Preview All Results
import pandas as pd
import os

report = {
    'Phase 1 -- No Defense':      'results/phase1_results.csv',
    'Phase 2a -- AT Defense':     'results/phase2_at_results.csv',
    'Phase 2b -- AT+KD Defense':  'results/phase2_atkd_results.csv',
}

display_cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']

for title, path in report.items():
    print()
    print('=' * 60)
    print(title)
    print('=' * 60)
    if not os.path.exists(path):
        print('  Not yet generated: ' + path)
        continue
    df = pd.read_csv(path)
    if df.empty:
        print('  File exists but has no data rows yet.')
        continue
    cols = [c for c in display_cols if c in df.columns]
    print(df[cols].to_string(index=False))
    print()
    print(f'  {len(df)} row(s) in {path}')

## Cell 11 — VRAM Monitor

Run this in a separate tab / kernel if you want to watch VRAM usage during a long experiment.

In [ ]:
# Cell 11 — VRAM Snapshot (run any time)
import torch

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    free, total = [x / 1e9 for x in torch.cuda.mem_get_info(0)]
    print(f'Allocated : {alloc:.2f} GB')
    print(f'Reserved  : {reserved:.2f} GB')
    print(f'Free      : {free:.2f} GB  /  {total:.2f} GB total')
else:
    print('No GPU.')

## Cell 12 — Checkpoint List

List all saved AT / AT+KD checkpoints and their sizes.

In [ ]:
# Cell 12 — List Checkpoints
import os

for ckpt_dir in ['results/checkpoints/at', 'results/checkpoints/atkd']:
    print(ckpt_dir + ':')
    if not os.path.isdir(ckpt_dir):
        print('  (directory not found)')
        continue
    files = sorted(os.listdir(ckpt_dir))
    if not files:
        print('  (empty — no checkpoints yet)')
    for f in files:
        size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
        print(f'  {f:<50}  {size_mb:6.1f} MB')
    print()

## Cell 13 — Generate Paper Figures

Generates all figures and Table 1 from the completed results CSVs.

Requires all phase CSVs to be present. Run after Cells 6–9 are complete.

**Outputs saved to `results/figures/`:**
- `fig1_attack_samples_{compression}_{attack}.png` — Original | Adversarial | Perturbation×10
- `fig2_asr_vs_epsilon_{compression}.png` — ASR vs ε curve (FGSM vs PGD)
- `fig3_defense_pipeline_{compression}_{attack}.png` — Pre-attack | Post-attack | Post-defense
- `fig4_compression_vs_defense.png` — Grouped bar chart: compression × defense effectiveness
- `table1_quantitative_metrics.csv` — Machine-readable metrics table
- `table1_quantitative_metrics.txt` — LaTeX `\begin{table}...\end{table}` ready to paste

**Estimated time:** ~20–30 min (model loads + attack generation for PSNR/SSIM).

In [ ]:
# Cell 13 — Generate Paper Figures
import subprocess, sys, os

SCRIPT = os.path.join('utils', 'paper_figures.py')

print('=' * 60)
print('Generating paper figures and Table 1')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, SCRIPT, '--n-samples', '4', '--n-eval', '200'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print()
print('[Figures] Exited with code ' + str(proc.returncode))

figures_dir = os.path.join('results', 'figures')
if os.path.isdir(figures_dir):
    files = sorted(f for f in os.listdir(figures_dir) if f != '.gitkeep')
    print(f'\n{len(files)} file(s) in {figures_dir}:')
    for f in files:
        size_kb = os.path.getsize(os.path.join(figures_dir, f)) / 1024
        print(f'  {f:<60}  {size_kb:6.0f} KB')